In [16]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [17]:
import sys
!{sys.executable} -m pip install sentence-transformers

In [18]:
# !pip install giotto-tda sentence-transformers pandas scikit-learn nltk

import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceLandscape

# Pobranie tokenizatora zdań
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/bamichal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [19]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/bamichal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [20]:
# 1. Wczytanie danych (załóżmy, że masz plik 'news_dataset.csv')
# df = pd.read_csv('news_dataset.csv')

# DO TESTÓW: Tworzymy mały zbiór zabawek (odkomentuj powyższe dla pełnych danych)
df = pd.DataFrame({
    'text': [
        "The quick brown fox jumps over the lazy dog. It was a sunny day.", 
        "Aliens have landed in New York. They are giving away free pizza. The government is hiding it.",
        "Stock markets closed higher today. Investors are optimistic about the upcoming quarter.",
        "Drinking bleach cures all diseases instantly! Doctors hate this one simple trick. Buy now."
    ],
    'label': [1, 0, 1, 0] # 1 - Real, 0 - Fake
})

# Inicjalizacja modelu językowego
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3) # Redukcja wymiaru dla wydajności TDA

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    if len(sentences) < 2:
        # Jeśli tekst jest za krótki, dodajemy sztuczne zdanie/zaburzenie, by stworzyć przestrzeń
        sentences.append(text + " additional context.")
        
    embeddings = model.encode(sentences)
    
    # Redukcja wymiaru do 3D
    if len(embeddings) >= 3:
        point_cloud = pca.fit_transform(embeddings)
    else:
        # Fallback jeśli zdań jest mało (PCA wymaga n_samples >= n_components)
        point_cloud = embeddings[:, :3] 
        
    return point_cloud

print("Generowanie chmur punktów...")
# Dla 45 tys. artykułów to zajmie trochę czasu! Warto zrobić to na mniejszej próbce.
point_clouds = [text_to_point_cloud(text) for text in df['text']]
y = df['label'].values

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generowanie chmur punktów...


In [21]:
print("Obliczanie homologii persystentnych i krajobrazów...")

# Konfiguracja Vietoris-Rips
# homology_dimensions=[0, 1] oznacza, że patrzymy na H0 i H1
vr = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=-1)

# Obliczenie diagramów persystencji (zwraca tablice 3D)
diagrams = vr.fit_transform(point_clouds)

# Konfiguracja Persistent Landscapes
# n_layers określa ile poziomów krajobrazu bierzemy pod uwagę
# n_bins to rozdzielczość wektora
pl = PersistenceLandscape(n_layers=5, n_bins=50)

# Przekształcenie diagramów w płaskie wektory (features)
X_topological = pl.fit_transform(diagrams)

print(f"Kształt macierzy cech topologicznych: {X_topological.shape}")

Obliczanie homologii persystentnych i krajobrazów...
Kształt macierzy cech topologicznych: (4, 10, 50)


In [22]:
# 1. Spłaszczenie macierzy 3D do 2D
# -1 oznacza "połącz wszystkie pozostałe wymiary w jeden"
n_samples = X_topological.shape[0]
X_topological_2d = X_topological.reshape(n_samples, -1)

print(f"Nowy kształt macierzy cech: {X_topological_2d.shape}")

# 2. Podział na zbiór treningowy i testowy (używamy teraz X_topological_2d)
X_train, X_test, y_train, y_test = train_test_split(
    X_topological_2d, y, test_size=0.2, random_state=42
)

# 3. Inicjalizacja i trening klasyfikatora
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 4. Predykcja i ewaluacja
y_pred = clf.predict(X_test)

print("Raport klasyfikacji na podstawie cech topologicznych:")
print(classification_report(y_test, y_pred))

Nowy kształt macierzy cech: (4, 500)
Raport klasyfikacji na podstawie cech topologicznych:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("Uruchamianie naiwnego baseline'u (TF-IDF)...")

# 1. Wektoryzacja tekstu (zamiana na macierz częstości słów)
# Używamy max_features, aby ograniczyć wymiarowość
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_naive = vectorizer.fit_transform(df['text'])

# 2. Podział na zbiór treningowy i testowy
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_naive, df['label'], test_size=0.2, random_state=42
)

# 3. Szybki trening klasyfikatora
clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_n, y_train_n)

# 4. Predykcja i ewaluacja
y_pred_n = clf_naive.predict(X_test_n)

print("Raport klasyfikacji dla naiwnego podejścia (TF-IDF + Logistic Regression):")
print(classification_report(y_test_n, y_pred_n))

Uruchamianie naiwnego baseline'u (TF-IDF)...
Raport klasyfikacji dla naiwnego podejścia (TF-IDF + Logistic Regression):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0



/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("mucahiddemircan/real-and-fake-news-dataset")

# print("Path to dataset files:", path)

In [ ]:
print(df.columns)

Index(['text'], dtype='str')


In [28]:
print(df_full.head(3))

                                                text  label
0  Gere faults Trump for blurring meaning of 'ref...      1
1  German parties start to find common ground in ...      1
2  Senate Democratic leader says Attorney General...      1


In [30]:
# import pandas as pd
# import numpy as np
# from nltk.tokenize import sent_tokenize
# from sentence_transformers import SentenceTransformer
# from sklearn.decomposition import PCA
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report
# from gtda.homology import VietorisRipsPersistence
# from gtda.diagrams import PersistenceLandscape

# print("Loading and cleaning data...")
# # Make sure the file name is 'news.csv'
# df_full = pd.read_csv('news.csv') 

# # CRITICAL STEP: Remove hidden whitespaces from column names just in case
# df_full.columns = df_full.columns.str.strip()

# # Sample 100 Real and 100 Fake articles
# print("Sampling data (100 Real, 100 Fake)...")
# df = df_full.groupby('label', group_keys=False).apply(lambda x: x.sample(100, random_state=42)).reset_index(drop=True)

# # ---------------------------------------------------------
# # PIPELINE 1: Naive (TF-IDF)
# # ---------------------------------------------------------
# print("\n[1/2] Running naive approach (TF-IDF)...")
# vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
# X_naive = vectorizer.fit_transform(df['text'])

# # ---------------------------------------------------------
# # PIPELINE 2: Topological (TDA)
# # ---------------------------------------------------------
# print("\n[2/2] Running topological approach (TDA)...")
# model = SentenceTransformer('all-MiniLM-L6-v2')
# pca = PCA(n_components=3)

# def text_to_point_cloud(text):
#     sentences = sent_tokenize(str(text))
    
#     # If text is too short, append noise to allow spatial representation
#     if len(sentences) < 2:
#         sentences.append(text + " additional context.")
        
#     embeddings = model.encode(sentences)
    
#     # Reduce dimensionality to 3D for performance
#     if len(embeddings) >= 3:
#         return pca.fit_transform(embeddings)
#     else:
#         return embeddings[:, :3]

# print("   -> Generating point clouds (Sentence Embeddings + PCA)...")
# point_clouds = [text_to_point_cloud(text) for text in df['text']]

# print("   -> Computing Vietoris-Rips complexes and landscapes...")
# vr = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=-1)
# diagrams = vr.fit_transform(point_clouds)

# pl = PersistenceLandscape(n_layers=5, n_bins=50)
# X_topological_3d = pl.fit_transform(diagrams)

# # Flatten 3D arrays to 2D for scikit-learn classifiers
# X_topological = X_topological_3d.reshape(X_topological_3d.shape[0], -1)

# # ---------------------------------------------------------
# # EVALUATION AND COMPARISON
# # ---------------------------------------------------------
# print("\nTraining models and generating reports...")
# y = df['label'].values

# # Split both feature sets at once to test on the exact same articles
# split = train_test_split(X_naive, X_topological, y, test_size=0.2, random_state=42)
# X_train_naive, X_test_naive, X_train_tda, X_test_tda, y_train, y_test = split

# # Train TF-IDF
# clf_naive = LogisticRegression(max_iter=1000)
# clf_naive.fit(X_train_naive, y_train)
# y_pred_naive = clf_naive.predict(X_test_naive)

# # Train TDA
# clf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
# clf_tda.fit(X_train_tda, y_train)
# y_pred_tda = clf_tda.predict(X_test_tda)

# print("\n" + "="*60)
# print("COMPARISON RESULTS (Test set: 40 articles)")
# print("="*60)

# print("\n--- NAIVE APPROACH (TF-IDF + Logistic Regression) ---")
# print(classification_report(y_test, y_pred_naive))

# print("\n--- TOPOLOGICAL APPROACH (TDA + Random Forest) ---")
# print(classification_report(y_test, y_pred_tda))

import pandas as pd
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceLandscape

print("Loading and cleaning data...")
# Make sure the file name is 'news.csv'
df_full = pd.read_csv('news.csv') 

# Strip hidden whitespaces from column names
df_full.columns = df_full.columns.str.strip()

# Modern, safe Pandas sampling method (avoids .apply() dropping the group column)
print("Sampling data (100 Real, 100 Fake)...")
df = df_full.groupby('label').sample(n=100, random_state=42).reset_index(drop=True)

# FAIL-FAST: Extract labels immediately to ensure they exist before heavy TDA computation
y = df['label'].values
print(f"Data ready. Target vector shape: {y.shape}")

# ---------------------------------------------------------
# PIPELINE 1: Naive (TF-IDF)
# ---------------------------------------------------------
print("\n[1/2] Running naive approach (TF-IDF)...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_naive = vectorizer.fit_transform(df['text'])

# ---------------------------------------------------------
# PIPELINE 2: Topological (TDA)
# ---------------------------------------------------------
print("\n[2/2] Running topological approach (TDA)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3)

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    
    # If text is too short, append noise to allow spatial representation
    if len(sentences) < 2:
        sentences.append(text + " additional context.")
        
    embeddings = model.encode(sentences)
    
    # Reduce dimensionality to 3D for performance
    if len(embeddings) >= 3:
        return pca.fit_transform(embeddings)
    else:
        return embeddings[:, :3]

print("   -> Generating point clouds (Sentence Embeddings + PCA)...")
point_clouds = [text_to_point_cloud(text) for text in df['text']]

print("   -> Computing Vietoris-Rips complexes and landscapes...")
vr = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=-1)
diagrams = vr.fit_transform(point_clouds)

pl = PersistenceLandscape(n_layers=5, n_bins=50)
X_topological_3d = pl.fit_transform(diagrams)

# Flatten 3D arrays to 2D for scikit-learn classifiers
X_topological = X_topological_3d.reshape(X_topological_3d.shape[0], -1)

# ---------------------------------------------------------
# EVALUATION AND COMPARISON
# ---------------------------------------------------------
print("\nTraining models and generating reports...")

# Split both feature sets at once to test on the exact same articles
split = train_test_split(X_naive, X_topological, y, test_size=0.2, random_state=42)
X_train_naive, X_test_naive, X_train_tda, X_test_tda, y_train, y_test = split

# Train TF-IDF
clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_naive, y_train)
y_pred_naive = clf_naive.predict(X_test_naive)

# Train TDA
clf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
clf_tda.fit(X_train_tda, y_train)
y_pred_tda = clf_tda.predict(X_test_tda)

print("\n" + "="*60)
print("COMPARISON RESULTS (Test set: 40 articles)")
print("="*60)

print("\n--- NAIVE APPROACH (TF-IDF + Logistic Regression) ---")
print(classification_report(y_test, y_pred_naive))

print("\n--- TOPOLOGICAL APPROACH (TDA + Random Forest) ---")
print(classification_report(y_test, y_pred_tda))

Loading and cleaning data...
Sampling data (100 Real, 100 Fake)...
Data ready. Target vector shape: (200,)

[1/2] Running naive approach (TF-IDF)...

[2/2] Running topological approach (TDA)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   -> Generating point clouds (Sentence Embeddings + PCA)...
   -> Computing Vietoris-Rips complexes and landscapes...

Training models and generating reports...

COMPARISON RESULTS (Test set: 40 articles)

--- NAIVE APPROACH (TF-IDF + Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.90      0.86      0.88        21
           1       0.85      0.89      0.87        19

    accuracy                           0.88        40
   macro avg       0.88      0.88      0.87        40
weighted avg       0.88      0.88      0.88        40


--- TOPOLOGICAL APPROACH (TDA + Random Forest) ---
              precision    recall  f1-score   support

           0       0.61      0.52      0.56        21
           1       0.55      0.63      0.59        19

    accuracy                           0.57        40
   macro avg       0.58      0.58      0.57        40
weighted avg       0.58      0.57      0.57        40



In [31]:
import pandas as pd
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceLandscape

print("Loading and cleaning data...")
df_full = pd.read_csv('news.csv') 
df_full.columns = df_full.columns.str.strip()

print("Sampling data (100 Real, 100 Fake)...")
df = df_full.groupby('label').sample(n=100, random_state=42).reset_index(drop=True)
y = df['label'].values

# ---------------------------------------------------------
# STEP 1: TF-IDF Features
# ---------------------------------------------------------
print("\n[1/3] Generating TF-IDF features...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
# We convert the sparse matrix to a dense array so we can concatenate it later
X_naive = vectorizer.fit_transform(df['text']).toarray() 

# ---------------------------------------------------------
# STEP 2: Topological Features (TDA)
# ---------------------------------------------------------
print("\n[2/3] Generating Topological features (TDA)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3)

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    if len(sentences) < 2:
        sentences.append(text + " additional context.")
    embeddings = model.encode(sentences)
    if len(embeddings) >= 3:
        return pca.fit_transform(embeddings)
    else:
        return embeddings[:, :3]

point_clouds = [text_to_point_cloud(text) for text in df['text']]
vr = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=-1)
diagrams = vr.fit_transform(point_clouds)

pl = PersistenceLandscape(n_layers=5, n_bins=50)
X_topological_3d = pl.fit_transform(diagrams)
X_topological = X_topological_3d.reshape(X_topological_3d.shape[0], -1)

# ---------------------------------------------------------
# STEP 3: The Ensemble (Combining Semantics + Topology)
# ---------------------------------------------------------
print("\n[3/3] Creating Ensemble features...")
# Horizontally stack the TF-IDF array and the TDA array
X_ensemble = np.hstack((X_naive, X_topological))

print(f"   -> TF-IDF shape: {X_naive.shape}")
print(f"   -> TDA shape: {X_topological.shape}")
print(f"   -> Final Ensemble shape: {X_ensemble.shape}")

# ---------------------------------------------------------
# EVALUATION AND COMPARISON
# ---------------------------------------------------------
print("\nTraining models and generating reports...")

# Split all three feature sets using the exact same random seed
X_train_n, X_test_n, X_train_t, X_test_t, X_train_e, X_test_e, y_train, y_test = train_test_split(
    X_naive, X_topological, X_ensemble, y, test_size=0.2, random_state=42
)

# 1. Train Naive Model
clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_n, y_train)
y_pred_naive = clf_naive.predict(X_test_n)

# 2. Train Pure TDA Model
clf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
clf_tda.fit(X_train_t, y_train)
y_pred_tda = clf_tda.predict(X_test_t)

# 3. Train ENSEMBLE Model
# We use Random Forest because it handles mixed feature types (frequencies + landscapes) very well
clf_ensemble = RandomForestClassifier(n_estimators=100, random_state=42)
clf_ensemble.fit(X_train_e, y_train)
y_pred_ensemble = clf_ensemble.predict(X_test_e)

print("\n" + "="*60)
print("FINAL COMPARISON RESULTS (Test set: 40 articles)")
print("="*60)

print("\n--- 1. PURE NAIWE (TF-IDF + Logistic Regression) ---")
print(classification_report(y_test, y_pred_naive))

print("\n--- 2. PURE TOPOLOGY (TDA + Random Forest) ---")
print(classification_report(y_test, y_pred_tda))

print("\n--- 3. THE ENSEMBLE (TF-IDF + TDA + Random Forest) ---")
print(classification_report(y_test, y_pred_ensemble))

Loading and cleaning data...
Sampling data (100 Real, 100 Fake)...

[1/3] Generating TF-IDF features...

[2/3] Generating Topological features (TDA)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[3/3] Creating Ensemble features...
   -> TF-IDF shape: (200, 1000)
   -> TDA shape: (200, 500)
   -> Final Ensemble shape: (200, 1500)

Training models and generating reports...

FINAL COMPARISON RESULTS (Test set: 40 articles)

--- 1. PURE NAIWE (TF-IDF + Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.90      0.86      0.88        21
           1       0.85      0.89      0.87        19

    accuracy                           0.88        40
   macro avg       0.88      0.88      0.87        40
weighted avg       0.88      0.88      0.88        40


--- 2. PURE TOPOLOGY (TDA + Random Forest) ---
              precision    recall  f1-score   support

           0       0.61      0.52      0.56        21
           1       0.55      0.63      0.59        19

    accuracy                           0.57        40
   macro avg       0.58      0.58      0.57        40
weighted avg       0.58      0.57      0.57        40


--- 3. THE EN

In [ ]:
import pandas as pd
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Importujemy czystego Ripsera
from ripser import ripser

print("Loading and cleaning data...")
df_full = pd.read_csv('news.csv') 
df_full.columns = df_full.columns.str.strip()

print("Sampling data (100 Real, 100 Fake)...")
df = df_full.groupby('label').sample(n=100, random_state=42).reset_index(drop=True)
y = df['label'].values

# ---------------------------------------------------------
# STEP 1: TF-IDF Features
# ---------------------------------------------------------
print("\n[1/3] Generating TF-IDF features...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_naive = vectorizer.fit_transform(df['text']).toarray() 

# ---------------------------------------------------------
# STEP 2: Topological Features (Using direct RIPSER)
# ---------------------------------------------------------
print("\n[2/3] Generating Topological features (Ripser)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3)

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    if len(sentences) < 2:
        sentences.append(text + " additional context.")
    embeddings = model.encode(sentences)
    if len(embeddings) >= 3:
        return pca.fit_transform(embeddings)
    else:
        return embeddings[:, :3]

print("   -> Generating point clouds...")
point_clouds = [text_to_point_cloud(text) for text in df['text']]

# --- RIPSER FEATURE EXTRACTION ---
def extract_ripser_features(point_cloud):
    # Ripser computes Vietoris-Rips and returns H0 and H1 by default (maxdim=1)
    # the 'dgms' key contains a list of arrays: [H0_diagram, H1_diagram]
    dgms = ripser(point_cloud, maxdim=1)['dgms']
    
    features = []
    
    # Loop over homology dimensions: 0 and 1
    for dim, dgm in enumerate(dgms):
        # Filter out features that live to infinity (always present in H0)
        dgm_finite = dgm[dgm[:, 1] != np.inf]
        
        if len(dgm_finite) == 0:
            # If no topological features exist, append zeros
            features.extend([0.0, 0.0, 0.0, 0.0])
            continue
            
        # Calculate lifespans (death - birth)
        lifespans = dgm_finite[:, 1] - dgm_finite[:, 0]
        
        # Extract meaningful statistics describing the shape's topology
        features.extend([
            len(lifespans),           # 1. Number of topological features (e.g., loops)
            np.max(lifespans),        # 2. Maximum persistence (most dominant feature)
            np.mean(lifespans),       # 3. Average persistence
            np.sum(lifespans)         # 4. Total persistence (similar to persistent entropy base)
        ])
    return features

print("   -> Computing Ripser barcodes and extracting statistics...")
# X_topological will now be a 2D array of shape (n_samples, 8 features) 
# (4 stats for H0 + 4 stats for H1)
X_topological = np.array([extract_ripser_features(pc) for pc in point_clouds])

# ---------------------------------------------------------
# STEP 3: The Ensemble (Combining Semantics + Topology)
# ---------------------------------------------------------
print("\n[3/3] Creating Ensemble features...")
X_ensemble = np.hstack((X_naive, X_topological))

print(f"   -> TF-IDF shape: {X_naive.shape}")
print(f"   -> Ripser Features shape: {X_topological.shape}")
print(f"   -> Final Ensemble shape: {X_ensemble.shape}")

# ---------------------------------------------------------
# EVALUATION AND COMPARISON
# ---------------------------------------------------------
print("\nTraining models and generating reports...")

X_train_n, X_test_n, X_train_t, X_test_t, X_train_e, X_test_e, y_train, y_test = train_test_split(
    X_naive, X_topological, X_ensemble, y, test_size=0.2, random_state=42
)

clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_n, y_train)
y_pred_naive = clf_naive.predict(X_test_n)

clf_tda = RandomForestClassifier(n_estimators=100, random_state=42)
clf_tda.fit(X_train_t, y_train)
y_pred_tda = clf_tda.predict(X_test_t)

clf_ensemble = RandomForestClassifier(n_estimators=100, random_state=42)
clf_ensemble.fit(X_train_e, y_train)
y_pred_ensemble = clf_ensemble.predict(X_test_e)

print("\n" + "="*60)
print("FINAL COMPARISON RESULTS (Test set: 40 articles)")
print("="*60)

print("\n--- 1. PURE NAIVE (TF-IDF + Logistic Regression) ---")
print(classification_report(y_test, y_pred_naive))

print("\n--- 2. PURE TOPOLOGY (Native Ripser Stats + Random Forest) ---")
print(classification_report(y_test, y_pred_tda))

print("\n--- 3. THE ENSEMBLE (TF-IDF + Ripser + Random Forest) ---")
print(classification_report(y_test, y_pred_ensemble))

# ---------------------------------------------------------
# ABLATION STUDY: TF-IDF + Random Forest
# ---------------------------------------------------------
print("\n--- 4. ABLATION STUDY (TF-IDF + Random Forest) ---")
clf_naive_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_naive_rf.fit(X_train_n, y_train)
y_pred_naive_rf = clf_naive_rf.predict(X_test_n)

print(classification_report(y_test, y_pred_naive_rf))

Loading and cleaning data...
Sampling data (100 Real, 100 Fake)...

[1/3] Generating TF-IDF features...

[2/3] Generating Topological features (Ripser)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   -> Generating point clouds...
   -> Computing Ripser barcodes and extracting statistics...

[3/3] Creating Ensemble features...
   -> TF-IDF shape: (200, 1000)
   -> Ripser Features shape: (200, 8)
   -> Final Ensemble shape: (200, 1008)

Training models and generating reports...


/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than row


FINAL COMPARISON RESULTS (Test set: 40 articles)

--- 1. PURE NAIVE (TF-IDF + Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.90      0.86      0.88        21
           1       0.85      0.89      0.87        19

    accuracy                           0.88        40
   macro avg       0.88      0.88      0.87        40
weighted avg       0.88      0.88      0.88        40


--- 2. PURE TOPOLOGY (Native Ripser Stats + Random Forest) ---
              precision    recall  f1-score   support

           0       0.62      0.38      0.47        21
           1       0.52      0.74      0.61        19

    accuracy                           0.55        40
   macro avg       0.57      0.56      0.54        40
weighted avg       0.57      0.55      0.54        40


--- 3. THE ENSEMBLE (TF-IDF + Ripser + Random Forest) ---
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        21
          

In [ ]:
import pandas as pd
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from ripser import ripser

print("Loading and cleaning data...")
df_full = pd.read_csv('news.csv') 
df_full.columns = df_full.columns.str.strip()

# --- HARD MODE: 5000 Articles (2500 Real, 2500 Fake) ---
print("Sampling data (2500 Real, 2500 Fake)... This might take a moment!")
# df = df_full.groupby('label').sample(n=2500, random_state=42).reset_index(drop=True)
df = df_full.groupby('label').sample(n=500, random_state=42).reset_index(drop=True)
y = df['label'].values

# ---------------------------------------------------------
# STEP 1: TF-IDF Features
# ---------------------------------------------------------
print("\n[1/3] Generating TF-IDF features...")
# Zwiększamy nieco max_features, aby model TF-IDF miał z czym pracować przy większym słowniku
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
X_naive = vectorizer.fit_transform(df['text']).toarray() 

# ---------------------------------------------------------
# STEP 2: Topological Features (Using direct RIPSER)
# ---------------------------------------------------------
print("\n[2/3] Generating Topological features (Ripser)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3)

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    if len(sentences) < 2:
        sentences.append(text + " additional context.")
    embeddings = model.encode(sentences)
    if len(embeddings) >= 3:
        return pca.fit_transform(embeddings)
    else:
        return embeddings[:, :3]

print("   -> Generating point clouds...")
point_clouds = [text_to_point_cloud(text) for text in df['text']]

def extract_ripser_features(point_cloud):
    dgms = ripser(point_cloud, maxdim=1)['dgms']
    features = []
    
    for dim, dgm in enumerate(dgms):
        dgm_finite = dgm[dgm[:, 1] != np.inf]
        
        if len(dgm_finite) == 0:
            features.extend([0.0, 0.0, 0.0, 0.0])
            continue
            
        lifespans = dgm_finite[:, 1] - dgm_finite[:, 0]
        
        features.extend([
            len(lifespans),           
            np.max(lifespans),        
            np.mean(lifespans),       
            np.sum(lifespans)         
        ])
    return features

print("   -> Computing Ripser barcodes and extracting statistics...")
# Ten krok może zająć od kilku do kilkunastu minut w zależności od Twojego procesora!
X_topological = np.array([extract_ripser_features(pc) for pc in point_clouds])

# ---------------------------------------------------------
# STEP 3: The Ensemble (Combining Semantics + Topology)
# ---------------------------------------------------------
print("\n[3/3] Creating Ensemble features...")
X_ensemble = np.hstack((X_naive, X_topological))

print(f"   -> TF-IDF shape: {X_naive.shape}")
print(f"   -> Ripser Features shape: {X_topological.shape}")
print(f"   -> Final Ensemble shape: {X_ensemble.shape}")

# ---------------------------------------------------------
# EVALUATION AND COMPARISON
# ---------------------------------------------------------
print("\nTraining models and generating reports...")

X_train_n, X_test_n, X_train_t, X_test_t, X_train_e, X_test_e, y_train, y_test = train_test_split(
    X_naive, X_topological, X_ensemble, y, test_size=0.2, random_state=42
)

# 1. Pure Naive (TF-IDF + LR)
clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_n, y_train)
y_pred_naive = clf_naive.predict(X_test_n)

# 2. Pure Topology (Ripser + RF)
clf_tda = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_tda.fit(X_train_t, y_train)
y_pred_tda = clf_tda.predict(X_test_t)

# 3. Ensemble (TF-IDF + Ripser + RF)
clf_ensemble = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_ensemble.fit(X_train_e, y_train)
y_pred_ensemble = clf_ensemble.predict(X_test_e)

# 4. Ablation Study (TF-IDF + RF)
clf_ablation = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_ablation.fit(X_train_n, y_train)
y_pred_ablation = clf_ablation.predict(X_test_n)

print("\n" + "="*60)
print(f"FINAL COMPARISON RESULTS (Test set: {len(y_test)} articles)")
print("="*60)

print("\n--- 1. PURE NAIVE (TF-IDF + Logistic Regression) ---")
print(classification_report(y_test, y_pred_naive))

print("\n--- 2. PURE TOPOLOGY (Native Ripser Stats + Random Forest) ---")
print(classification_report(y_test, y_pred_tda))

print("\n--- 3. THE ENSEMBLE (TF-IDF + Ripser + Random Forest) ---")
print(classification_report(y_test, y_pred_ensemble))

print("\n--- 4. ABLATION STUDY (TF-IDF + Random Forest) ---")
print(classification_report(y_test, y_pred_ablation))

Loading and cleaning data...
Sampling data (2500 Real, 2500 Fake)... This might take a moment!

[1/3] Generating TF-IDF features...

[2/3] Generating Topological features (Ripser)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   -> Generating point clouds...
   -> Computing Ripser barcodes and extracting statistics...


/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than rows; did you mean to transpose?
  warnings.warn(
/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/ripser/ripser.py:257: UserWarning: The input point cloud has more columns than row


[3/3] Creating Ensemble features...
   -> TF-IDF shape: (1000, 3000)
   -> Ripser Features shape: (1000, 8)
   -> Final Ensemble shape: (1000, 3008)

Training models and generating reports...

FINAL COMPARISON RESULTS (Test set: 200 articles)

--- 1. PURE NAIVE (TF-IDF + Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.91      0.90      0.90        96
           1       0.90      0.91      0.91       104

    accuracy                           0.91       200
   macro avg       0.91      0.90      0.90       200
weighted avg       0.91      0.91      0.90       200


--- 2. PURE TOPOLOGY (Native Ripser Stats + Random Forest) ---
              precision    recall  f1-score   support

           0       0.57      0.61      0.59        96
           1       0.62      0.58      0.60       104

    accuracy                           0.59       200
   macro avg       0.60      0.60      0.59       200
weighted avg       0.60      0.59      

: 